# Combine Multiple Single-Category Masks into One Multi-Category Mask

> For every source image there are several binary masks, one per category. This notebook combines those masks into a single multi-category mask, keeps the original image unchanged, and saves the pair as a new dataset ready for SAM3 fine-tuning.


In [ ]:
#| default_exp preprocessing.combine_multicategory

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pathlib import Path
import json
import os
from typing import Dict, List, Tuple, Union
from fastcore.all import *

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

## 1. Build Category Mapping

Create a COCO-style category list from human-readable names. Category ids are 1-based.

In [ ]:
#| export
def build_categories(category_names: List[str]) -> List[Dict]:
    """Build a COCO-style category list from ordered category names.

    Args:
        category_names: Ordered list of category names.

    Returns:
        List of dicts with keys 'id' (1-based) and 'name'.
    """
    return [
        {"id": i + 1, "name": name}
        for i, name in enumerate(category_names)
    ]

In [ ]:
build_categories(['thick_wire', 'thin_wire', 'gatepads'])

## 2. Load Source Image and Masks

Read the original image and the binary masks for each category using OpenCV. The image is returned RGB; masks are 2D uint8 arrays.

In [ ]:
#| export
def load_image(path: Union[str, Path]) -> np.ndarray:
    """Load an image with OpenCV and convert BGR -> RGB."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def load_mask(path: Union[str, Path]) -> np.ndarray:
    """Load a binary mask as a 2D uint8 numpy array."""
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Could not load mask: {path}")
    return mask

## 3. Discover Image / Multi-Mask Pairs

For the Bent-Wire dataset the masks share the image stem and use category suffixes such as `lblthin`, `lblthick`, and `lblgatepads`. This helper discovers every image together with its masks.

In [ ]:
#| export
def discover_bentwire_pairs(
    image_dir: Union[str, Path],
    mask_suffixes: List[str] = None,
    image_extensions: Tuple[str, ...] = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"),
) -> List[Dict]:
    """Discover image / multi-mask pairs in a Bent-Wire-style folder.

    For every original image found in `image_dir`, this looks for masks that
    share the same stem plus a category suffix, e.g.:
        image:      IMG_001_original.png
        masks:      IMG_001_lblthin.png
                    IMG_001_lblthick.png
                    IMG_001_lblgatepads.png

    Args:
        image_dir: Directory containing original images and masks.
        mask_suffixes: Ordered list of mask suffixes / category names.
            Defaults to ['lblthin', 'lblthick', 'lblgatepads'].
        image_extensions: File extensions considered as images.

    Returns:
        List of dicts, one per image, with keys:
            - 'image': path to the original image
            - 'masks': list of mask paths (same order as mask_suffixes)
            - 'category_names': same as mask_suffixes
    """
    if mask_suffixes is None:
        mask_suffixes = ["lblthin", "lblthick", "lblgatepads"]

    image_dir = Path(image_dir)
    assert image_dir.is_dir(), f"Directory not found: {image_dir}"

    exts = set(image_extensions)
    all_files = [p for p in image_dir.iterdir() if p.is_file()]
    mask_suffixes_lower = [s.lower() for s in mask_suffixes]

    def is_mask_path(p: Path) -> bool:
        stem_lower = p.stem.lower()
        return any(stem_lower.endswith(s) for s in mask_suffixes_lower)

    image_paths = [
        p for p in all_files
        if p.suffix.lower() in exts and not is_mask_path(p) and 'original' in p.stem.lower()
    ]
    image_paths.sort(key=lambda p: p.name)

    pairs = []
    for img_path in image_paths:
        masks = []
        for suffix in mask_suffixes:
            mask_path = img_path.parent / f"{img_path.stem.replace('original', '')}_{suffix}{img_path.suffix}"
            if not mask_path.exists():
                mask_path = img_path.parent / f"{img_path.stem.replace('original', '')}{suffix}{img_path.suffix}"
            masks.append(mask_path)
        pairs.append({
            "image": img_path,
            "masks": masks,
            "category_names": mask_suffixes,
        })

    return pairs

In [ ]:

base_dir = Path('/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Train/20250512_parted_thin/20250512_parted_thin_Labelled_part2')
output_dir = Path('/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin_multicat')

pairs = discover_bentwire_pairs(base_dir, mask_suffixes=["lblthin", "lblthick", "gatepads"])

In [ ]:
pairs[1]['image']

In [ ]:
pairs[1]['masks']

## 4. Visualize Source Image and Masks

Display the single original image and all category masks to verify the data before combining.

In [ ]:
#| export
def plot_image_and_masks(
    image: np.ndarray,
    masks: List[np.ndarray],
    category_names: List[str],
    figsize: Tuple[int, int] = (16, 4),
):
    """Display one source image and its category masks side-by-side."""
    n = len(masks)
    fig, axes = plt.subplots(1, n + 1, figsize=figsize)

    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    for i, (mask, name) in enumerate(zip(masks, category_names)):
        axes[i + 1].imshow(mask, cmap="gray")
        axes[i + 1].set_title(f"Mask: {name}")
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_image_and_masks(
    image=load_image(pairs[0]['image']),
    masks=[load_mask(p) for p in pairs[0]['masks']],
    category_names=pairs[0]['category_names'],
    figsize=(16, 4),
)

## 5. Assign Category Labels to Each Mask

Convert each binary mask into a labeled mask where foreground pixels are replaced with the corresponding 1-based category id.

In [ ]:
#| export
def label_masks(masks: List[np.ndarray], category_ids: List[int]) -> List[np.ndarray]:
    """Convert binary masks into labeled masks with unique category values.

    Args:
        masks: List of binary masks (H, W) with foreground > 0.
        category_ids: Integer category id for each mask.

    Returns:
        List of labeled masks where foreground pixels equal category_id.
    """
    labeled = []
    for mask, cat_id in zip(masks, category_ids):
        labeled_mask = np.where(mask > 0, int(cat_id), 0).astype(np.uint8)
        labeled.append(labeled_mask)
    return labeled

In [ ]:
lbled_msks = label_masks(
    masks=[load_mask(p) for p in pairs[0]['masks']],
    category_ids=[1, 2, 3],
)

## 6. Combine Masks into a Single Multi-Category Mask

Overlay the labeled masks into one mask array. Overlapping regions can be resolved by priority order, maximum category id, or first-wins.

In [ ]:
#| export
def combine_masks(
    labeled_masks: List[np.ndarray],
    overlap_rule: str = "priority",
) -> np.ndarray:
    """Overlay labeled masks into one multi-category mask.

    Args:
        labeled_masks: List of masks with unique category ids.
        overlap_rule: How to handle overlapping foreground pixels.
            - "priority": later masks overwrite earlier ones.
            - "max": keep the highest category id.
            - "first": first mask wins.

    Returns:
        A single (H, W) uint8 mask with category ids.
    """
    if not labeled_masks:
        raise ValueError("labeled_masks must not be empty")

    h, w = labeled_masks[0].shape
    combined = np.zeros((h, w), dtype=np.uint8)

    for mask in labeled_masks:
        fg = mask > 0
        if overlap_rule == "priority":
            combined[fg] = mask[fg]
        elif overlap_rule == "max":
            combined = np.maximum(combined, mask)
        elif overlap_rule == "first":
            combined = np.where((combined == 0) & fg, mask, combined)
        else:
            raise ValueError(f"Unknown overlap_rule: {overlap_rule}")

    return combined

In [ ]:
cobined_mask = combine_masks(
    labeled_masks=label_masks(
        masks=[load_mask(p) for p in pairs[0]['masks']],
        category_ids=[1, 2, 3],
    ),
    overlap_rule="priority",
)

In [ ]:
plt.imshow(cobined_mask)

## 7. Save the Image and Multi-Category Mask

Write the original image and two mask files:
- `*_multicat_mask.png` — raw category ids (1, 2, 3, ...) for model training / COCO conversion.
- `*_multicat_mask_vis.png` — scaled to the full 0-255 range so a human can see the different categories.

In [ ]:
#| export
def save_image_and_mask(
    image: np.ndarray,
    combined_mask: np.ndarray,
    output_dir: Path,
    stem: str = "sample",
) -> Dict[str, Path]:
    """Write the image and multi-category mask to disk.

    Saves:
        - <stem>_multicat.png          : original image
        - <stem>_multicat_mask.png     : mask with raw category ids
        - <stem>_multicat_mask_vis.png : human-visible mask scaled to 0-255

    Returns:
        Dictionary with saved image, raw mask, and visible mask paths.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_path = output_dir / f"{stem}_multicat.png"
    mask_path = output_dir / f"{stem}_multicat_mask.png"
    vis_mask_path = output_dir / f"{stem}_multicat_mask_vis.png"

    cv2.imwrite(str(image_path), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(mask_path), combined_mask)

    # Human-visible mask: scale category ids evenly across 0-255
    n_cats = int(combined_mask.max())
    if n_cats > 0:
        vis_mask = (combined_mask.astype(np.uint16) * 255 // n_cats).astype(np.uint8)
    else:
        vis_mask = combined_mask.copy()
    cv2.imwrite(str(vis_mask_path), vis_mask)

    return {"image": image_path, "mask": mask_path, "vis_mask": vis_mask_path}

## 8. Create Dataset Metadata

Generate a JSON file that records the image filename, mask filename, category labels, and the overlap rule used.

In [ ]:
#| export
def build_dataset_metadata(
    output_dir: Path,
    categories: List[Dict],
    image_name: str,
    mask_name: str,
    overlap_rule: str,
    image_size: Tuple[int, int],
    colors: Dict[int, Tuple[int, int, int]] = None,
) -> Path:
    """Generate a JSON metadata file describing the combined dataset."""
    metadata = {
        "image_file": image_name,
        "mask_file": mask_name,
        "image_width": image_size[0],
        "image_height": image_size[1],
        "categories": categories,
        "overlap_rule": overlap_rule,
        "colors": {str(k): v for k, v in colors.items()} if colors else {},
    }

    meta_path = output_dir / "metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    return meta_path

## 9. Verify the Saved Outputs

Reload the saved image and mask, overlay the mask on the image with a color map, and display the result to confirm the multi-category dataset was created correctly.

In [ ]:
#| export
def verify_output(
    image_path: Union[str, Path],
    mask_path: Union[str, Path],
    categories: List[Dict],
    alpha: float = 0.5,
    figsize: Tuple[int, int] = (14, 6),
):
    """Reload saved image/mask, overlay the mask, and display the result."""
    img = load_image(image_path)
    mask = load_mask(mask_path)

    n_cats = len(categories)
    cmap = plt.cm.get_cmap("tab10", max(n_cats, 2))
    overlay = np.zeros_like(img, dtype=np.float32)

    for cat in categories:
        cid = cat["id"]
        color = np.array(cmap(cid - 1)[:3]) * 255
        overlay[mask == cid] = color

    blended = (img * (1 - alpha) + overlay * alpha).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    axes[0].imshow(img)
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="nipy_spectral", vmin=0, vmax=n_cats)
    axes[1].set_title("Multi-Category Mask")
    axes[1].axis("off")

    axes[2].imshow(blended)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    print("Category ids present in mask:", sorted(np.unique(mask).tolist()))

## 10. End-to-End Helpers

`combine_one_image_multi_masks` processes a single image together with its category masks. `process_folder` runs this over every discovered pair and is the usual entry point.

In [ ]:
#| export
def combine_one_image_multi_masks(
    image_path: Union[str, Path],
    mask_paths: List[Union[str, Path]],
    category_names: List[str],
    output_dir: Union[str, Path],
    overlap_rule: str = "priority",
    verify: bool = False,
) -> Dict:
    """Build a single multi-category mask for one image from its category masks.

    The original image is copied unchanged; only the masks are combined into a
    single multi-category PNG.

    Args:
        image_path: Path to the original image.
        mask_paths: List of binary mask paths (one per category).
        category_names: Category name for each mask.
        output_dir: Where to save the output image, mask, and metadata.
        overlap_rule: Passed to combine_masks().
        verify: If True, display the saved image/mask overlay.

    Returns:
        Dictionary with saved image_path, mask_path, and metadata_path.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    assert len(mask_paths) == len(category_names), \
        "mask_paths and category_names must have the same length."

    categories = build_categories(category_names)
    image = load_image(image_path)
    masks = [load_mask(p) for p in mask_paths]

    plot_image_and_masks(image, masks, category_names)

    labeled_masks = label_masks(masks, [c["id"] for c in categories])
    combined_mask = combine_masks(labeled_masks, overlap_rule=overlap_rule)

    stem = Path(image_path).stem
    saved = save_image_and_mask(image, combined_mask, output_dir, stem=stem)

    h, w = image.shape[:2]
    meta_path = build_dataset_metadata(
        output_dir=output_dir,
        categories=categories,
        image_name=saved["image"].name,
        mask_name=saved["mask"].name,
        overlap_rule=overlap_rule,
        image_size=(w, h),
    )

    if verify:
        verify_output(saved["image"], saved["mask"], categories)

    return {
        "image_path": saved["image"],
        "mask_path": saved["mask"],
        "metadata_path": meta_path,
    }


def process_folder(
    image_dir: Union[str, Path],
    output_dir: Union[str, Path],
    mask_suffixes: List[str] = None,
    category_names: List[str] = None,
    overlap_rule: str = "priority",
    verify_first: bool = False,
) -> List[Dict]:
    """Process every image/multi-mask pair in a Bent-Wire-style folder.

    Args:
        image_dir: Folder containing original images and masks.
        output_dir: Folder where combined image/mask pairs are saved.
        mask_suffixes: Suffixes used to discover masks (defaults to
            ['lblthin', 'lblthick', 'lblgatepads']).
        category_names: Final category names to write into metadata. If None,
            mask_suffixes are used.
        overlap_rule: Passed to combine_masks().
        verify_first: If True, run verify_output() on the first sample.

    Returns:
        List of result dicts, one per processed image.
    """
    if mask_suffixes is None:
        mask_suffixes = ["lblthin", "lblthick", "lblgatepads"]
    if category_names is None:
        category_names = mask_suffixes

    pairs = discover_bentwire_pairs(image_dir, mask_suffixes=mask_suffixes)
    if not pairs:
        print(f"No image/mask pairs found in {image_dir}")
        return []

    results = []
    for i, pair in enumerate(tqdm(pairs, desc="Combining masks")):
        result = combine_one_image_multi_masks(
            image_path=pair["image"],
            mask_paths=pair["masks"],
            category_names=category_names,
            output_dir=output_dir,
            overlap_rule=overlap_rule,
            verify=False,#(verify_first and i == 0),
        )
        results.append(result)

    return results

## Example Usage — Bent-Wire folder layout

For the HPC folder `/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin/` the masks share the image stem and add suffixes `lblthin`, `lblthick`, and `lblgatepads`. The cell below discovers all pairs and creates one multi-category mask per image.

In [ ]:
#base_dir = Path('/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Train/20250512_parted_thin/20250512_parted_thin_Labelled_part2')
base_dir_ = Path('/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test')
#base_dir_ = Path('/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Train/20250507_N50_FS980_FS01ENG_FS01_FS02_FS03/FS01ENG')
base_dir_list = base_dir_.ls()
output_dir = Path('/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin_multicat_test')
Path(output_dir).mkdir(parents=True, exist_ok=True)
for i in base_dir_list:
    if i.is_dir():
        if Path(i).name in ['Thin_multicat_test', 'Thin_multicat', 'Thin']:
            continue

        print(i)
        results = process_folder(
        image_dir=i,
        output_dir=output_dir,
        mask_suffixes=['lblthick', 'lblthin', 'gatepads'], # last wins *** important for priority overlap rule
        category_names=['Thick', 'Thin', 'Gatepads'],
        overlap_rule='priority',
        verify_first=False,
    )

### Single-image quick test

If you want to test on just one image/mask group before running the whole folder, use `combine_one_image_multi_masks` directly.

In [ ]:
# result = combine_one_image_multi_masks(
#     image_path='/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin/SomeImage.png',
#     mask_paths=[
#         '/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin/SomeImage_lblthin.png',
#         '/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin/SomeImage_lblthick.png',
#         '/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin/SomeImage_lblgatepads.png',
#     ],
#     category_names=['Thin', 'Thick', 'Gatepads'],
#     output_dir='/home/hpc_storage/data/projects/userB/data/bent-wire/Bent-Wire-ALU-Test/Thin_multicat',
#     overlap_rule='priority',
#     verify=True,
# )
# print(result)